In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(train.shape)
print(test.shape)

(699635, 23)
(299844, 22)


In [2]:
y = train["satisfaction"].astype(int)

X = train.drop(columns=["satisfaction"])
X_test = test.copy()

print(y.mean())

0.44357272006117476


In [3]:
def add_source_features(df):
    df = df.copy()

    # ----------------------------------------
    # 1. Digital Experience
    # ----------------------------------------
    df["digital_score"] = (
        df["Inflight wifi service"] +
        df["Ease of Online booking"] +
        df["Online boarding"]
    )

    # ----------------------------------------
    # 2. Passenger Comfort
    # ----------------------------------------
    df["comfort_score"] = (
        df["Food and drink"] +
        df["Seat comfort"] +
        df["Inflight entertainment"] +
        df["Leg room service"]
    )

    # ----------------------------------------
    # 3. Service Quality
    # ----------------------------------------
    df["service_score"] = (
        df["On-board service"] +
        df["Baggage handling"] +
        df["Checkin service"] +
        df["Cleanliness"]
    )

    # ----------------------------------------
    # 4. Important interactions
    # ----------------------------------------

    # Class × digital experience
    df["class_digital"] = (
        df["Class"].astype(str)
        + "_" +
        df["digital_score"].astype(str)
    )

    # Class × online boarding
    df["class_boarding"] = (
        df["Class"].astype(str)
        + "_" +
        df["Online boarding"].astype(str)
    )

    # Travel type × boarding
    df["travel_boarding"] = (
        df["Type of Travel"].astype(str)
        + "_" +
        df["Online boarding"].astype(str)
    )

    # Customer type × boarding
    df["customer_boarding"] = (
        df["Customer Type"].astype(str)
        + "_" +
        df["Online boarding"].astype(str)
    )

    return df

In [4]:

source = pd.read_csv("new Dataset/train.csv")

In [5]:
X_source = add_source_features(X)
X_test_source = add_source_features(X_test)

print("Original:", X.shape)
print("New:", X_source.shape)

new_features = [
    "digital_score",
    "comfort_score",
    "service_score",
    "class_digital",
    "class_boarding",
    "travel_boarding",
    "customer_boarding"
]

print(X_source[new_features].head())

Original: (699635, 22)
New: (699635, 29)
   digital_score  comfort_score  service_score class_digital class_boarding  \
0             12              7             11        Eco_12          Eco_4   
1             14             19             17   Business_14     Business_4   
2              8             12             16         Eco_8          Eco_3   
3              9             14             14         Eco_9          Eco_3   
4              9             19             18         Eco_9          Eco_3   

     travel_boarding    customer_boarding  
0  Personal Travel_4     Loyal Customer_4  
1  Business travel_4     Loyal Customer_4  
2  Personal Travel_3     Loyal Customer_3  
3  Personal Travel_3     Loyal Customer_3  
4  Business travel_3  disloyal Customer_3  


In [8]:
# ============================================================
# FIX CATEGORICAL DTYPES
# ============================================================

X_source = X_source.copy()
X_test_source = X_test_source.copy()

categorical_cols = X_source.select_dtypes(
    include=["object"]
).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

for col in categorical_cols:
    X_source[col] = X_source[col].astype("category")
    X_test_source[col] = X_test_source[col].astype("category")

print("\nRemaining object columns:")
print(
    X_source.select_dtypes(include=["object"]).columns.tolist()
)

Categorical columns:
[]

Remaining object columns:
[]


In [9]:
from sklearn.model_selection import train_test_split

train_idx, val_idx = train_test_split(
    np.arange(len(y)),
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", len(train_idx))
print("Validation:", len(val_idx))

Train: 559708
Validation: 139927


In [6]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_source,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
from lightgbm import LGBMClassifier, early_stopping
from sklearn.metrics import roc_auc_score

source_model = LGBMClassifier(
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=127,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=0.5,
    random_state=42,
    n_jobs=-1
)

source_model.fit(
    X_source.iloc[train_idx],
    y.iloc[train_idx],
    eval_set=[
        (
            X_source.iloc[val_idx],
            y.iloc[val_idx]
        )
    ],
    eval_metric="auc",
    callbacks=[
        early_stopping(100, verbose=False)
    ]
)

source_pred = source_model.predict_proba(
    X_source.iloc[val_idx]
)[:, 1]

source_auc = roc_auc_score(
    y.iloc[val_idx],
    source_pred
)

print("=" * 60)
print("SOURCE-INFORMED FEATURES")
print("=" * 60)
print("Best iteration:", source_model.best_iteration_)
print("AUC:", source_auc)

C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.031659 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 996
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 29
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
SOURCE-INFORMED FEATURES
Best iteration: 576
AUC: 0.9578202667820336


In [11]:
X_base = train.drop(columns=["satisfaction"]).copy()

categorical_base = X_base.select_dtypes(
    include=["object"]
).columns.tolist()

for col in categorical_base:
    X_base[col] = X_base[col].astype("category")

In [12]:
baseline_model = LGBMClassifier(
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=127,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=0.5,
    random_state=42,
    n_jobs=-1
)

baseline_model.fit(
    X_base.iloc[train_idx],
    y.iloc[train_idx],
    eval_set=[
        (
            X_base.iloc[val_idx],
            y.iloc[val_idx]
        )
    ],
    eval_metric="auc",
    callbacks=[
        early_stopping(100, verbose=False)
    ]
)

baseline_pred = baseline_model.predict_proba(
    X_base.iloc[val_idx]
)[:, 1]

baseline_auc = roc_auc_score(
    y.iloc[val_idx],
    baseline_pred
)

print("=" * 60)
print("BASELINE")
print("=" * 60)
print("Best iteration:", baseline_model.best_iteration_)
print("AUC:", baseline_auc)

C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.032381 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 852
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 22
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
BASELINE
Best iteration: 539
AUC: 0.9578858735200408


In [13]:
print("\n" + "=" * 60)
print("FINAL COMPARISON")
print("=" * 60)

print(f"Baseline AUC       : {baseline_auc:.9f}")
print(f"Source-feature AUC : {source_auc:.9f}")
print(f"Delta              : {source_auc - baseline_auc:+.9f}")


FINAL COMPARISON
Baseline AUC       : 0.957885874
Source-feature AUC : 0.957820267
Delta              : -0.000065607


## CATBoost

In [15]:
source.shape

(103904, 25)

In [16]:
# ============================================================
# EXP — EXTERNAL AIRLINE DATASET MODEL
# Source dataset -> S6E10 validation
# ============================================================

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("EXTERNAL SOURCE MODEL EXPERIMENT")
print("=" * 70)


# ============================================================
# 1. LOAD SOURCE DATA
# ============================================================

# SOURCE_PATH = "/kaggle/input/airline-passenger-satisfaction/Airline Passenger Satisfaction.csv"

# source = pd.read_csv(SOURCE_PATH)

print("Source shape:", source.shape)


# ============================================================
# 2. PREPARE TARGET
# ============================================================

source["satisfaction"] = (
    source["satisfaction"]
    .map({
        "satisfied": 1,
        "neutral or dissatisfied": 0
    })
)

print("\nSource target:")
print(source["satisfaction"].value_counts(normalize=True))


# ============================================================
# 3. IDENTIFY COMPETITION FEATURES
# ============================================================

target_col = "satisfaction"
id_col = "id"

# Competition predictors
feature_cols = [
    "Gender",
    "Customer Type",
    "Age",
    "Type of Travel",
    "Class",
    "Flight Distance",
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes"
]

# IMPORTANT:
# "Inflight service" is deliberately excluded because
# it does not exist in the competition dataset.


# ============================================================
# 4. CHECK ALIGNMENT
# ============================================================

missing_source = [
    c for c in feature_cols
    if c not in source.columns
]

print("\nMissing source features:", missing_source)

assert len(missing_source) == 0, \
    f"Missing features: {missing_source}"


# ============================================================
# 5. BUILD SOURCE TRAINING DATA
# ============================================================

X_source = source[feature_cols].copy()
y_source = source[target_col].astype(int)

# Competition validation data
X_comp_val = X_val[feature_cols].copy()

print("\nSource X:", X_source.shape)
print("Source y:", y_source.shape)
print("Competition validation X:", X_comp_val.shape)


# ============================================================
# 6. HANDLE MISSING VALUES
# ============================================================

# CatBoost handles numerical NaNs natively.
# Fill categorical missing values explicitly.

categorical_cols = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class"
]

for col in categorical_cols:
    X_source[col] = X_source[col].fillna("Missing").astype(str)
    X_comp_val[col] = X_comp_val[col].fillna("Missing").astype(str)


# ============================================================
# 7. TRAIN SOURCE MODEL
# ============================================================

cat_features = [
    X_source.columns.get_loc(c)
    for c in categorical_cols
]

model_source = CatBoostClassifier(
    iterations=1200,
    learning_rate=0.03,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200,
    thread_count=-1,
    l2_leaf_reg=5
)

model_source.fit(
    X_source,
    y_source,
    cat_features=cat_features
)


# ============================================================
# 8. PREDICT S6E10 VALIDATION
# ============================================================

source_val_pred = model_source.predict_proba(
    X_comp_val
)[:, 1]

print("\nPrediction shape:", source_val_pred.shape)
print("Min:", source_val_pred.min())
print("Max:", source_val_pred.max())
print("Mean:", source_val_pred.mean())


# ============================================================
# 9. SOURCE MODEL AUC ON COMPETITION VALIDATION
# ============================================================

source_auc = roc_auc_score(
    y_val,
    source_val_pred
)

print("\n" + "=" * 70)
print("SOURCE MODEL RESULT")
print("=" * 70)

print(f"External source AUC: {source_auc:.9f}")

EXTERNAL SOURCE MODEL EXPERIMENT
Source shape: (103904, 25)

Source target:
satisfaction
0    0.566667
1    0.433333
Name: proportion, dtype: float64

Missing source features: []

Source X: (103904, 21)
Source y: (103904,)
Competition validation X: (139927, 21)
0:	total: 323ms	remaining: 6m 27s
200:	total: 32.2s	remaining: 2m 40s
400:	total: 1m 3s	remaining: 2m 6s
600:	total: 1m 34s	remaining: 1m 34s
800:	total: 2m 6s	remaining: 1m 3s
1000:	total: 2m 38s	remaining: 31.6s
1199:	total: 3m 12s	remaining: 0us

Prediction shape: (139927,)
Min: 9.895720334823248e-07
Max: 0.9999998848025557
Mean: 0.459446332887762

SOURCE MODEL RESULT
External source AUC: 0.953880178


In [19]:
# ============================================================
# FIND VALIDATION PREDICTIONS
# ============================================================

import numpy as np

prediction_candidates = []

for name in dir():
    try:
        value = eval(name)
        
        if isinstance(value, np.ndarray):
            if value.ndim == 1 and len(value) == len(y_val):
                prediction_candidates.append(
                    (name, value.shape, value.dtype)
                )
    except:
        pass

for item in prediction_candidates:
    print(item)

('baseline_pred', (139927,), dtype('float64'))
('source_pred', (139927,), dtype('float64'))
('source_val_pred', (139927,), dtype('float64'))
('val_idx', (139927,), dtype('int64'))
('value', (139927,), dtype('int64'))


In [20]:
# ============================================================
# EXTERNAL MODEL — DIVERSITY + BLEND TEST
# ============================================================

from sklearn.metrics import roc_auc_score
import numpy as np

champion_val_pred = baseline_pred
external_val_pred = source_val_pred

# ------------------------------------------------------------
# 1. Individual AUCs
# ------------------------------------------------------------

champion_auc = roc_auc_score(y_val, champion_val_pred)
external_auc = roc_auc_score(y_val, external_val_pred)

# ------------------------------------------------------------
# 2. Prediction correlation
# ------------------------------------------------------------

corr = np.corrcoef(
    champion_val_pred,
    external_val_pred
)[0, 1]

print("=" * 70)
print("EXTERNAL MODEL DIVERSITY TEST")
print("=" * 70)

print(f"Champion AUC : {champion_auc:.9f}")
print(f"External AUC : {external_auc:.9f}")
print(f"Correlation  : {corr:.9f}")


# ------------------------------------------------------------
# 3. Search external-model weight
# ------------------------------------------------------------

best_auc = champion_auc
best_weight = 0.0

for external_weight in np.arange(0.00, 0.51, 0.01):

    champion_weight = 1.0 - external_weight

    blend_pred = (
        champion_weight * champion_val_pred
        + external_weight * external_val_pred
    )

    auc = roc_auc_score(y_val, blend_pred)

    if auc > best_auc:
        best_auc = auc
        best_weight = external_weight


# ------------------------------------------------------------
# 4. Result
# ------------------------------------------------------------

improvement = best_auc - champion_auc

print("\n" + "=" * 70)
print("BLEND RESULT")
print("=" * 70)

print(f"Champion AUC       : {champion_auc:.9f}")
print(f"Best blend AUC     : {best_auc:.9f}")
print(f"Improvement        : {improvement:+.9f}")
print(f"External weight    : {best_weight:.2f}")
print(f"Champion weight    : {1-best_weight:.2f}")

if improvement > 0:
    print("\n🔥 EXTERNAL MODEL ADDS DIVERSITY")
else:
    print("\n❌ EXTERNAL MODEL DOES NOT IMPROVE")

EXTERNAL MODEL DIVERSITY TEST
Champion AUC : 0.957885874
External AUC : 0.953880178
Correlation  : 0.990744748

BLEND RESULT
Champion AUC       : 0.957885874
Best blend AUC     : 0.957962493
Improvement        : +0.000076620
External weight    : 0.19
Champion weight    : 0.81

🔥 EXTERNAL MODEL ADDS DIVERSITY


In [27]:
# Find all prediction arrays of validation length
import numpy as np

for name in dir():
    try:
        value = eval(name)
        if isinstance(value, np.ndarray) and value.ndim == 1:
            if len(value) == len(y_val):
                auc = roc_auc_score(y_val, value)
                if auc > 0.95:
                    print(f"{name:40s} AUC = {auc:.9f}")
    except:
        pass

baseline_pred                            AUC = 0.957885874
blend_pred                               AUC = 0.957813627
champion_val_pred                        AUC = 0.957885874
external_val_pred                        AUC = 0.953880178
source_pred                              AUC = 0.957820267
source_val_pred                          AUC = 0.953880178


In [24]:
import os

TABICL_DIR = "artifacts/tabicl_test"

print("Files in TabICL folder:")
for f in os.listdir(TABICL_DIR):
    path = os.path.join(TABICL_DIR, f)
    print(f"{f:40s} {os.path.getsize(path) / 1024**2:.2f} MB")

Files in TabICL folder:
.ipynb_checkpoints                       0.00 MB
tabicl_100k_test.npy                     1.14 MB
tabicl_50k_test.npy                      1.14 MB


In [34]:
tabicl_val_pred = np.load(
    "artifacts/oof/tabicl_100k_val.npy"
)

print(tabicl_val_pred.shape)

(139927,)


In [35]:
# ============================================================
# TABICL 100K — DIVERSITY ANALYSIS
# ============================================================

from sklearn.metrics import roc_auc_score
import numpy as np

# Use the prediction you just generated

# ------------------------------------------------------------
# 1. Standalone AUCs
# ------------------------------------------------------------

baseline_auc = roc_auc_score(y_val, baseline_pred)
tabicl_auc = roc_auc_score(y_val, tabicl_val_pred)
external_auc = roc_auc_score(y_val, source_val_pred)

print("=" * 70)
print("STANDALONE VALIDATION AUC")
print("=" * 70)

print(f"Baseline      : {baseline_auc:.9f}")
print(f"TabICL 100K   : {tabicl_auc:.9f}")
print(f"External      : {external_auc:.9f}")


# ------------------------------------------------------------
# 2. Prediction correlations
# ------------------------------------------------------------

pred_df = np.column_stack([
    baseline_pred,
    tabicl_val_pred,
    source_val_pred
])

corr = np.corrcoef(pred_df, rowvar=False)

print("\n" + "=" * 70)
print("PREDICTION CORRELATION")
print("=" * 70)

print(
    "                 Baseline    TabICL    External"
)

print(
    f"Baseline         1.000000    "
    f"{corr[0,1]:.6f}    {corr[0,2]:.6f}"
)

print(
    f"TabICL           {corr[1,0]:.6f}    "
    f"1.000000    {corr[1,2]:.6f}"
)

print(
    f"External         {corr[2,0]:.6f}    "
    f"{corr[2,1]:.6f}    1.000000"
)

STANDALONE VALIDATION AUC
Baseline      : 0.957885874
TabICL 100K   : 0.956320376
External      : 0.953880178

PREDICTION CORRELATION
                 Baseline    TabICL    External
Baseline         1.000000    0.995125    0.990745
TabICL           0.995125    1.000000    0.990333
External         0.990745    0.990333    1.000000


In [36]:
# ============================================================
# TABICL 100K — 2-WAY BLEND SEARCH
# ============================================================

from sklearn.metrics import roc_auc_score
import numpy as np

base_auc = roc_auc_score(y_val, baseline_pred)

best_auc = base_auc
best_weight = 0.0

results = []

# Focus on small TabICL weights because TabICL is weaker
for tabicl_weight in np.arange(0.00, 0.31, 0.005):

    blend_pred = (
        (1.0 - tabicl_weight) * baseline_pred
        + tabicl_weight * tabicl_val_pred
    )

    auc = roc_auc_score(y_val, blend_pred)

    results.append((tabicl_weight, auc))

    if auc > best_auc:
        best_auc = auc
        best_weight = tabicl_weight

print("=" * 70)
print("BASELINE + TABICL 100K")
print("=" * 70)

print(f"Baseline AUC : {base_auc:.9f}")
print(f"Best AUC     : {best_auc:.9f}")
print(f"Improvement  : {best_auc - base_auc:+.9f}")
print(f"TabICL weight: {best_weight:.3f}")
print(f"Base weight  : {1-best_weight:.3f}")

BASELINE + TABICL 100K
Baseline AUC : 0.957885874
Best AUC     : 0.957987116
Improvement  : +0.000101242
TabICL weight: 0.195
Base weight  : 0.805


In [37]:
# ============================================================
# 3-WAY BLEND
# Baseline + TabICL + External Source
# ============================================================

from sklearn.metrics import roc_auc_score
import numpy as np

print("=" * 70)
print("3-WAY DIVERSITY BLEND")
print("=" * 70)

base = baseline_pred
tabicl = tabicl_val_pred
external = source_val_pred

base_auc = roc_auc_score(y_val, base)

best_auc = base_auc
best_weights = (1.0, 0.0, 0.0)

# Search around the useful regions we already discovered.
#
# TabICL optimum ≈ 19.5%
# External optimum ≈ 19%
#
# We'll search total auxiliary weight <= 40%.

for tabicl_weight in np.arange(0.00, 0.301, 0.005):

    for external_weight in np.arange(0.00, 0.301, 0.005):

        base_weight = 1.0 - tabicl_weight - external_weight

        # Invalid if auxiliary models exceed 100%
        if base_weight < 0:
            continue

        blend = (
            base_weight * base
            + tabicl_weight * tabicl
            + external_weight * external
        )

        auc = roc_auc_score(y_val, blend)

        if auc > best_auc:
            best_auc = auc
            best_weights = (
                base_weight,
                tabicl_weight,
                external_weight
            )

print("\n" + "=" * 70)
print("BEST 3-WAY BLEND")
print("=" * 70)

print(f"Baseline AUC : {base_auc:.9f}")
print(f"Best AUC     : {best_auc:.9f}")
print(f"Improvement  : {best_auc - base_auc:+.9f}")

print("\nWeights:")
print(f"Baseline : {best_weights[0]:.3f}")
print(f"TabICL   : {best_weights[1]:.3f}")
print(f"External : {best_weights[2]:.3f}")

3-WAY DIVERSITY BLEND

BEST 3-WAY BLEND
Baseline AUC : 0.957885874
Best AUC     : 0.958011864
Improvement  : +0.000125990

Weights:
Baseline : 0.720
TabICL   : 0.150
External : 0.130


In [38]:
# ============================================================
# 3-WAY BLEND — LOCAL FINE SEARCH
# ============================================================

from sklearn.metrics import roc_auc_score
import numpy as np

base = baseline_pred
tabicl = tabicl_val_pred
external = source_val_pred

best_auc = -1
best_weights = None

results = []

# Local search around 72 / 15 / 13
for tabicl_weight in np.arange(0.10, 0.201, 0.001):

    for external_weight in np.arange(0.08, 0.181, 0.001):

        base_weight = 1.0 - tabicl_weight - external_weight

        if base_weight < 0:
            continue

        blend = (
            base_weight * base
            + tabicl_weight * tabicl
            + external_weight * external
        )

        auc = roc_auc_score(y_val, blend)

        results.append(
            (auc, base_weight, tabicl_weight, external_weight)
        )

        if auc > best_auc:
            best_auc = auc
            best_weights = (
                base_weight,
                tabicl_weight,
                external_weight
            )

print("=" * 70)
print("3-WAY BLEND — LOCAL FINE SEARCH")
print("=" * 70)

print(f"Best AUC     : {best_auc:.9f}")
print(f"Improvement  : {best_auc - roc_auc_score(y_val, base):+.9f}")

print("\nBest weights:")
print(f"Baseline : {best_weights[0]:.3f}")
print(f"TabICL   : {best_weights[1]:.3f}")
print(f"External : {best_weights[2]:.3f}")

# Top 10 combinations
results = sorted(results, reverse=True)

print("\nTop 10:")
for auc, bw, tw, ew in results[:10]:
    print(
        f"AUC={auc:.9f} | "
        f"Base={bw:.3f} | "
        f"TabICL={tw:.3f} | "
        f"Ext={ew:.3f}"
    )

3-WAY BLEND — LOCAL FINE SEARCH
Best AUC     : 0.958011933
Improvement  : +0.000126059

Best weights:
Baseline : 0.719
TabICL   : 0.150
External : 0.131

Top 10:
AUC=0.958011933 | Base=0.719 | TabICL=0.150 | Ext=0.131
AUC=0.958011891 | Base=0.717 | TabICL=0.150 | Ext=0.133
AUC=0.958011876 | Base=0.718 | TabICL=0.150 | Ext=0.132
AUC=0.958011871 | Base=0.718 | TabICL=0.149 | Ext=0.133
AUC=0.958011868 | Base=0.719 | TabICL=0.149 | Ext=0.132
AUC=0.958011864 | Base=0.720 | TabICL=0.150 | Ext=0.130
AUC=0.958011843 | Base=0.716 | TabICL=0.155 | Ext=0.129
AUC=0.958011842 | Base=0.723 | TabICL=0.149 | Ext=0.128
AUC=0.958011842 | Base=0.718 | TabICL=0.151 | Ext=0.131
AUC=0.958011839 | Base=0.717 | TabICL=0.151 | Ext=0.132


## PHASE 6A — DELAY FEATURE RESULTS

In [46]:
# ============================================================
# PHASE 6A — DELAY RELATIONSHIP FEATURES
# ============================================================

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, early_stopping
from sklearn.metrics import roc_auc_score


def add_delay_features(df):
    df = df.copy()

    dep = df["Departure Delay in Minutes"].fillna(0)
    arr = df["Arrival Delay in Minutes"].fillna(0)

    # Delay relationships
    df["total_delay"] = dep + arr
    df["delay_difference"] = arr - dep
    df["abs_delay_difference"] = (arr - dep).abs()

    # Delay states
    df["dep_delayed"] = (dep > 0).astype("int8")
    df["arr_delayed"] = (arr > 0).astype("int8")
    df["any_delay"] = ((dep > 0) | (arr > 0)).astype("int8")
    df["both_delayed"] = ((dep > 0) & (arr > 0)).astype("int8")

    return df


# ------------------------------------------------------------
# Remove OLD source-informed interaction features
# ------------------------------------------------------------

OLD_FEATURES = [
    "class_digital",
    "class_boarding",
    "travel_boarding",
    "customer_boarding"
]

X_tr_delay = X_tr.drop(
    columns=OLD_FEATURES,
    errors="ignore"
).copy()

X_val_delay = X_val.drop(
    columns=OLD_FEATURES,
    errors="ignore"
).copy()


# ------------------------------------------------------------
# Add ONLY delay features
# ------------------------------------------------------------

X_tr_delay = add_delay_features(X_tr_delay)
X_val_delay = add_delay_features(X_val_delay)


# ------------------------------------------------------------
# Categorical columns
# ------------------------------------------------------------

cat_cols = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class"
]

for col in cat_cols:
    X_tr_delay[col] = X_tr_delay[col].astype("category")
    X_val_delay[col] = X_val_delay[col].astype("category")


# ------------------------------------------------------------
# Safety check
# ------------------------------------------------------------

print("Object columns:",
      X_tr_delay.select_dtypes(include=["object"]).columns.tolist())

print("Train shape:", X_tr_delay.shape)
print("Validation shape:", X_val_delay.shape)


# ------------------------------------------------------------
# LightGBM
# ------------------------------------------------------------

delay_model = LGBMClassifier(
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=127,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=0.5,
    random_state=42,
    n_jobs=-1
)

delay_model.fit(
    X_tr_delay,
    y_tr,
    eval_set=[(X_val_delay, y_val)],
    eval_metric="auc",
    categorical_feature=cat_cols,
    callbacks=[
        early_stopping(100, verbose=False)
    ]
)


# ------------------------------------------------------------
# Evaluate
# ------------------------------------------------------------

delay_pred = delay_model.predict_proba(X_val_delay)[:, 1]

delay_auc = roc_auc_score(y_val, delay_pred)

BASELINE_AUC = 0.957885874

print("\n" + "=" * 60)
print("PHASE 6A — DELAY FEATURE RESULTS")
print("=" * 60)

print(f"Baseline AUC  : {BASELINE_AUC:.9f}")
print(f"Delay AUC     : {delay_auc:.9f}")
print(f"Delta         : {delay_auc - BASELINE_AUC:+.9f}")
print(f"Best iteration: {delay_model.best_iteration_}")

Object columns: []
Train shape: (559708, 32)
Validation shape: (139927, 32)


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.037929 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1188
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 32
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676

PHASE 6A — DELAY FEATURE RESULTS
Baseline AUC  : 0.957885874
Delay AUC     : 0.957749611
Delta         : -0.000136263
Best iteration: 515


## Phase 6B — Service Consistency Features

In [48]:
# ============================================================
# PHASE 6B — SERVICE CONSISTENCY FEATURES
# ============================================================

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, early_stopping
from sklearn.metrics import roc_auc_score


# ------------------------------------------------------------
# Service / experience rating columns
# ------------------------------------------------------------

SERVICE_COLS = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness"
]


# ------------------------------------------------------------
# Feature creation
# ------------------------------------------------------------

def add_service_consistency_features(df):
    df = df.copy()

    service = df[SERVICE_COLS]

    # Overall passenger experience
    df["service_mean"] = service.mean(axis=1)
    df["service_std"] = service.std(axis=1)
    df["service_min"] = service.min(axis=1)
    df["service_max"] = service.max(axis=1)
    df["service_range"] = (
        df["service_max"] - df["service_min"]
    )

    # Rating distribution
    df["service_low_count"] = (
        service <= 2
    ).sum(axis=1)

    df["service_high_count"] = (
        service >= 4
    ).sum(axis=1)

    return df


# ------------------------------------------------------------
# Remove old experimental interaction features
# ------------------------------------------------------------

OLD_FEATURES = [
    "class_digital",
    "class_boarding",
    "travel_boarding",
    "customer_boarding"
]

X_tr_service = X_tr.drop(
    columns=OLD_FEATURES,
    errors="ignore"
).copy()

X_val_service = X_val.drop(
    columns=OLD_FEATURES,
    errors="ignore"
).copy()


# ------------------------------------------------------------
# Add service consistency features
# ------------------------------------------------------------

X_tr_service = add_service_consistency_features(X_tr_service)
X_val_service = add_service_consistency_features(X_val_service)


# ------------------------------------------------------------
# Categorical columns
# ------------------------------------------------------------

cat_cols = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class"
]

for col in cat_cols:
    X_tr_service[col] = X_tr_service[col].astype("category")
    X_val_service[col] = X_val_service[col].astype("category")


# ------------------------------------------------------------
# Safety check
# ------------------------------------------------------------

print("Object columns:",
      X_tr_service.select_dtypes(include=["object"]).columns.tolist())

print("Train shape:", X_tr_service.shape)
print("Validation shape:", X_val_service.shape)


# ------------------------------------------------------------
# LightGBM
# ------------------------------------------------------------

service_model = LGBMClassifier(
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=127,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=0.5,
    random_state=42,
    n_jobs=-1
)

service_model.fit(
    X_tr_service,
    y_tr,
    eval_set=[(X_val_service, y_val)],
    eval_metric="auc",
    categorical_feature=cat_cols,
    callbacks=[
        early_stopping(100, verbose=False)
    ]
)


# ------------------------------------------------------------
# Evaluate
# ------------------------------------------------------------

service_pred = service_model.predict_proba(
    X_val_service
)[:, 1]

service_auc = roc_auc_score(
    y_val,
    service_pred
)

BASELINE_AUC = 0.957885874

print("\n" + "=" * 60)
print("PHASE 6B — SERVICE CONSISTENCY RESULTS")
print("=" * 60)

print(f"Baseline AUC  : {BASELINE_AUC:.9f}")
print(f"Service AUC   : {service_auc:.9f}")
print(f"Delta         : {service_auc - BASELINE_AUC:+.9f}")
print(f"Best iteration: {service_model.best_iteration_}")

Object columns: []
Train shape: (559708, 32)
Validation shape: (139927, 32)


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.069487 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1233
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 32
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676

PHASE 6B — SERVICE CONSISTENCY RESULTS
Baseline AUC  : 0.957885874
Service AUC   : 0.957658433
Delta         : -0.000227441
Best iteration: 431


## Phase 6C — Segment Interaction Features

In [49]:
# ============================================================
# PHASE 6C — SEGMENT INTERACTION FEATURES
# ============================================================

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, early_stopping
from sklearn.metrics import roc_auc_score


# ------------------------------------------------------------
# Clean copies — remove previous experimental features
# ------------------------------------------------------------

OLD_FEATURES = [
    "class_digital",
    "class_boarding",
    "travel_boarding",
    "customer_boarding",

    # In case any previous Phase 6 features were added
    "total_delay",
    "delay_difference",
    "abs_delay_difference",
    "dep_delayed",
    "arr_delayed",
    "any_delay",
    "both_delayed",

    "service_mean",
    "service_std",
    "service_min",
    "service_max",
    "service_range",
    "service_low_count",
    "service_high_count"
]

X_tr_seg = X_tr.drop(
    columns=OLD_FEATURES,
    errors="ignore"
).copy()

X_val_seg = X_val.drop(
    columns=OLD_FEATURES,
    errors="ignore"
).copy()


# ------------------------------------------------------------
# Segment interaction features
# ------------------------------------------------------------

# Class × Type of Travel
X_tr_seg["class_travel"] = (
    X_tr_seg["Class"].astype(str)
    + "_"
    + X_tr_seg["Type of Travel"].astype(str)
)

X_val_seg["class_travel"] = (
    X_val_seg["Class"].astype(str)
    + "_"
    + X_val_seg["Type of Travel"].astype(str)
)


# Class × Customer Type
X_tr_seg["class_customer"] = (
    X_tr_seg["Class"].astype(str)
    + "_"
    + X_tr_seg["Customer Type"].astype(str)
)

X_val_seg["class_customer"] = (
    X_val_seg["Class"].astype(str)
    + "_"
    + X_val_seg["Customer Type"].astype(str)
)


# Age group
def make_age_group(df):
    return pd.cut(
        df["Age"],
        bins=[-np.inf, 25, 35, 50, 65, np.inf],
        labels=["<=25", "26-35", "36-50", "51-65", "66+"]
    ).astype(str)


X_tr_seg["age_group"] = make_age_group(X_tr_seg)
X_val_seg["age_group"] = make_age_group(X_val_seg)


# Class × Age Group
X_tr_seg["class_age"] = (
    X_tr_seg["Class"].astype(str)
    + "_"
    + X_tr_seg["age_group"]
)

X_val_seg["class_age"] = (
    X_val_seg["Class"].astype(str)
    + "_"
    + X_val_seg["age_group"]
)


# Type of Travel × Customer Type
X_tr_seg["travel_customer"] = (
    X_tr_seg["Type of Travel"].astype(str)
    + "_"
    + X_tr_seg["Customer Type"].astype(str)
)

X_val_seg["travel_customer"] = (
    X_val_seg["Type of Travel"].astype(str)
    + "_"
    + X_val_seg["Customer Type"].astype(str)
)


# ------------------------------------------------------------
# Categorical columns
# ------------------------------------------------------------

cat_cols = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class",
    "class_travel",
    "class_customer",
    "age_group",
    "class_age",
    "travel_customer"
]

for col in cat_cols:
    X_tr_seg[col] = X_tr_seg[col].astype("category")
    X_val_seg[col] = X_val_seg[col].astype("category")


# ------------------------------------------------------------
# Safety check
# ------------------------------------------------------------

print("Object columns:",
      X_tr_seg.select_dtypes(include=["object"]).columns.tolist())

print("Train shape:", X_tr_seg.shape)
print("Validation shape:", X_val_seg.shape)

print("\nInteraction distributions:")
for col in [
    "class_travel",
    "class_customer",
    "class_age",
    "travel_customer"
]:
    print(f"{col}: {X_tr_seg[col].nunique()} categories")


# ------------------------------------------------------------
# LightGBM
# ------------------------------------------------------------

segment_model = LGBMClassifier(
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=127,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=0.5,
    random_state=42,
    n_jobs=-1
)

segment_model.fit(
    X_tr_seg,
    y_tr,
    eval_set=[(X_val_seg, y_val)],
    eval_metric="auc",
    categorical_feature=cat_cols,
    callbacks=[
        early_stopping(100, verbose=False)
    ]
)


# ------------------------------------------------------------
# Evaluate
# ------------------------------------------------------------

segment_pred = segment_model.predict_proba(
    X_val_seg
)[:, 1]

segment_auc = roc_auc_score(
    y_val,
    segment_pred
)

BASELINE_AUC = 0.957885874

print("\n" + "=" * 60)
print("PHASE 6C — SEGMENT INTERACTION RESULTS")
print("=" * 60)

print(f"Baseline AUC  : {BASELINE_AUC:.9f}")
print(f"Segment AUC   : {segment_auc:.9f}")
print(f"Delta         : {segment_auc - BASELINE_AUC:+.9f}")
print(f"Best iteration: {segment_model.best_iteration_}")

Object columns: []
Train shape: (559708, 30)
Validation shape: (139927, 30)

Interaction distributions:
class_travel: 6 categories
class_customer: 6 categories
class_age: 15 categories
travel_customer: 4 categories


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023626 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 943
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 30
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676

PHASE 6C — SEGMENT INTERACTION RESULTS
Baseline AUC  : 0.957885874
Segment AUC   : 0.957865203
Delta         : -0.000020671
Best iteration: 426
